In [ ]:
# CELDA 1: Instalar
!pip install requests beautifulsoup4 pandas lxml -q
print('Instalado!')

In [ ]:
# CELDA 2: Funciones MEJORADAS
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from urllib.parse import quote
import random

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0',
    'Accept': 'text/html',
    'Accept-Language': 'en-US,en;q=0.9',
}
session = requests.Session()
session.headers.update(headers)

def extraer_apellido(nombre):
    if pd.isna(nombre):
        return None
    nombre = str(nombre).strip()
    apellido = re.sub(r'^([A-Z]\.\s*)+', '', nombre).strip()
    if not apellido:
        apellido = nombre
    # Si tiene espacios, tomar la primera palabra del apellido
    partes = apellido.split()
    return partes[0] if partes else apellido

def normalizar(texto):
    if not texto:
        return ''
    texto = texto.lower()
    for k, v in {'\u00e1':'a','\u00e9':'e','\u00ed':'i','\u00f3':'o','\u00fa':'u','\u00f1':'n','\u00fc':'u'}.items():
        texto = texto.replace(k, v)
    return texto

def buscar_jugador(nombre_original, equipo=None):
    try:
        apellido = extraer_apellido(nombre_original)
        if not apellido:
            return None, None
        
        apellido_norm = normalizar(apellido)
        url = 'https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche?query=' + quote(apellido)
        r = session.get(url, timeout=15)
        if r.status_code != 200:
            return None, None
        
        soup = BeautifulSoup(r.content, 'lxml')
        tables = soup.find_all('table', class_='items')
        if not tables:
            return None, None
        
        rows = tables[0].find_all('tr', class_=['odd', 'even'])
        mejor_match = None
        mejor_score = 0
        
        equipo_norm = normalizar(equipo) if equipo else ''
        palabras_equipo = [p for p in equipo_norm.split() if len(p) > 3]
        
        for row in rows:
            for link in row.find_all('a'):
                href = link.get('href', '')
                if '/profil/spieler/' in href:
                    player_url = 'https://www.transfermarkt.com' + href
                    player_name = link.get_text(strip=True)
                    player_name_norm = normalizar(player_name)
                    row_text = normalizar(row.get_text())
                    
                    # VALIDACION 1: El apellido debe estar en el nombre del jugador
                    if apellido_norm not in player_name_norm:
                        break  # Saltar este jugador
                    
                    score = 0
                    
                    # VALIDACION 2: Verificar equipo
                    for palabra in palabras_equipo:
                        if palabra in row_text:
                            score += 3
                            break
                    
                    # Bonus: pais sudamericano
                    paises = ['colombia', 'argentina', 'uruguay', 'paraguay', 'chile', 'peru', 'ecuador', 'venezuela']
                    for pais in paises:
                        if pais in row_text:
                            score += 1
                            break
                    
                    if score > mejor_score:
                        mejor_score = score
                        mejor_match = {'url': player_url, 'nombre': player_name}
                    
                    break
        
        # Solo devolver si encontro equipo (score >= 3)
        if mejor_match and mejor_score >= 3:
            return mejor_match['url'], mejor_match['nombre']
        
        return None, None
    except Exception as e:
        return None, None

def obtener_datos(url):
    datos = {
        'transfermarkt_url': url,
        'nombre_tm': None,
        'valor_mercado': None,
        'agente': None,
        'fin_contrato': None,
        'imagen_url': None
    }
    try:
        r = session.get(url, timeout=15)
        if r.status_code != 200:
            return datos
        
        soup = BeautifulSoup(r.content, 'lxml')
        
        # NOMBRE
        h1 = soup.find('h1', class_='data-header__headline-wrapper')
        if h1:
            datos['nombre_tm'] = h1.get_text(strip=True)
        
        # VALOR DE MERCADO
        mv = soup.find('a', class_='data-header__market-value-wrapper')
        if mv:
            datos['valor_mercado'] = mv.get_text(strip=True)
        
        # IMAGEN
        img = soup.find('img', class_='data-header__profile-image')
        if img:
            datos['imagen_url'] = img.get('src')
        
        # AGENTE - buscar en varias ubicaciones
        agent_link = soup.find('a', href=re.compile(r'/berater/'))
        if agent_link:
            datos['agente'] = agent_link.get_text(strip=True)
        
        if not datos['agente']:
            # Buscar en la seccion de info
            info_table = soup.find('div', class_='info-table')
            if info_table:
                agent_a = info_table.find('a', href=re.compile(r'berater'))
                if agent_a:
                    datos['agente'] = agent_a.get_text(strip=True)
        
        # FIN DE CONTRATO - buscar en varias ubicaciones
        page_text = soup.get_text()
        
        # Patron 1: Contract expires: Jun 30, 2025
        contract = re.search(r'Contract expires:\s*(\w+\s+\d+,\s+\d{4})', page_text)
        if contract:
            datos['fin_contrato'] = contract.group(1)
        
        # Patron 2: Contract until: Jun 30, 2025
        if not datos['fin_contrato']:
            contract = re.search(r'Contract until:\s*(\w+\s+\d+,\s+\d{4})', page_text)
            if contract:
                datos['fin_contrato'] = contract.group(1)
        
        # Patron 3: Solo el ano
        if not datos['fin_contrato']:
            contract = re.search(r'Contract expires:\s*(\d{4})', page_text)
            if contract:
                datos['fin_contrato'] = contract.group(1)
        
        return datos
    except:
        return datos

print('Funciones cargadas! (con validacion de apellido)')

In [ ]:
# CELDA 3: Cargar CSV
csv_base = 'https://docs.google.com/spreadsheets/d/e/'
csv_id = '2PACX-1vSneBjGlw2I3SyXV-uw1V8Cs_O4lbiQw39melKEZJNhunpshakPrn7AZQBN2L8N9Yw_HA-EeVOt3qvf'
csv_params = '/pub?gid=2002620668&single=true&output=csv'
CSV_URL = csv_base + csv_id + csv_params

df = pd.read_csv(CSV_URL)
print(f'Jugadores cargados: {len(df)}')
df[['Jugador', 'Equipo', 'Liga']].head(10)

In [ ]:
# CELDA 4: CONFIGURACION
DESDE = 0
HASTA = 50
PAUSA_MIN = 3
PAUSA_MAX = 5

total = min(HASTA, len(df)) - DESDE
print(f'Procesara {total} jugadores')
print(f'Tiempo estimado: {total * 4 / 60:.1f} minutos')

In [ ]:
# CELDA 5: EJECUTAR SCRAPING
resultados = []
total = min(HASTA, len(df)) - DESDE

print(f'Procesando {total} jugadores...')
print('Valida: equipo + apellido en nombre')
print('=' * 60)

for idx in range(DESDE, min(HASTA, len(df))):
    row = df.iloc[idx]
    nombre = row['Jugador']
    equipo = row.get('Equipo', '')
    liga = row.get('Liga', '')
    
    print(f'[{idx+1-DESDE}/{total}] {nombre} ({equipo})...', end=' ')
    
    url, nombre_tm = buscar_jugador(nombre, equipo)
    
    if url:
        datos = obtener_datos(url)
        datos['jugador_csv'] = nombre
        datos['equipo_csv'] = equipo
        datos['liga_csv'] = liga
        resultados.append(datos)
        print(f'OK -> {nombre_tm}')
        if datos['valor_mercado']:
            print(f'         Valor: {datos["valor_mercado"]}')
    else:
        print('NO ENCONTRADO')
        resultados.append({
            'jugador_csv': nombre,
            'equipo_csv': equipo,
            'liga_csv': liga,
            'transfermarkt_url': None,
            'nombre_tm': None,
            'valor_mercado': None,
            'agente': None,
            'fin_contrato': None,
            'imagen_url': None
        })
    
    time.sleep(random.uniform(PAUSA_MIN, PAUSA_MAX))

print('=' * 60)
encontrados = sum(1 for r in resultados if r['transfermarkt_url'])
print(f'Encontrados: {encontrados}/{total} ({encontrados/total*100:.0f}%)')

In [ ]:
# CELDA 6: Ver resultados encontrados
df_result = pd.DataFrame(resultados)

# Reordenar columnas
cols = ['jugador_csv', 'nombre_tm', 'equipo_csv', 'liga_csv', 'valor_mercado', 'fin_contrato', 'agente', 'transfermarkt_url', 'imagen_url']
df_result = df_result[[c for c in cols if c in df_result.columns]]

print('ENCONTRADOS:')
encontrados_df = df_result[df_result['transfermarkt_url'].notna()]
print(f'Total: {len(encontrados_df)}')
display(encontrados_df)

In [ ]:
# CELDA 7: Guardar y descargar
archivo = f'transfermarkt_{DESDE}_{HASTA}.csv'
df_result.to_csv(archivo, index=False, encoding='utf-8-sig')
print(f'Guardado: {archivo}')

from google.colab import files
files.download(archivo)

In [ ]:
# CELDA 8: Ver no encontrados
no_encontrados = df_result[df_result['transfermarkt_url'].isna()]
print(f'No encontrados: {len(no_encontrados)}')
print('\nProbablemente no estan en Transfermarkt o el equipo no coincide:')
for _, r in no_encontrados.head(50).iterrows():
    print(f"  - {r['jugador_csv']} ({r['equipo_csv']})")